In [1]:
import sys
import os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve().parent))

from sedi.consciousness_receiver import consciousness_scan, calibrate_null
from sedi.seti_scanner import full_scan, scan_breakthrough_listen
from sedi.sources.quantum_rng import fetch_quantum_random
from sedi.sources.exoplanet import fetch_habitable_zone
from sedi.constants import N, SIGMA, TAU, PHI, SOPFR

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)

print("SEDI engine loaded. Starting Layer 1 scan...")

SEDI engine loaded. Starting Layer 1 scan...


In [2]:
print("Calibrating null model (100 trials)...")
null_dist = calibrate_null(n_trials=100, data_length=5000)
for k, v in null_dist.items():
    print(f"  {k:20s}  mean={v['mean']:.4f}  std={v['std']:.4f}")

Calibrating null model (100 trials)...


  kuramoto              mean=0.6012  std=0.0095
  phi                   mean=0.3708  std=0.0984
  tension               mean=0.0000  std=0.0000
  golden_zone           mean=0.0000  std=0.0000
  channels              mean=0.0160  std=0.0731
  birth                 mean=3.3035  std=4.1442
  dedekind              mean=0.0000  std=0.0000
  attractor             mean=0.5207  std=0.0884


In [3]:
results = []

# 1. White noise control
rng = np.random.default_rng(42)
noise = rng.uniform(0, 255, size=5000)
r = consciousness_scan(noise, source_name='White Noise Control', calibrated=True)
results.append(r)
print(f"White Noise: {r['level']} (n_detected={r['n_detected']}/8)")

# 2. Quantum RNG (ANU)
qdata = fetch_quantum_random(1024)
if qdata:
    q = np.array(qdata, dtype=float)
    r = consciousness_scan(q, source_name='ANU Quantum RNG', calibrated=True)
    results.append(r)
    print(f"ANU QRNG: {r['level']} (n_detected={r['n_detected']}/8)")
else:
    print("ANU API unavailable — skipping")

# 3. Classical PRNG
urandom = rng.uniform(0, 65535, size=5000)
r = consciousness_scan(urandom, source_name='/dev/urandom (classical)', calibrated=True)
results.append(r)
print(f"/dev/urandom: {r['level']} (n_detected={r['n_detected']}/8)")

# 4. Voyager 1 (documented values — actual filterbank analysis too large for notebook)
results.append({
    'source_name': 'BL Voyager 1 Band 1',
    'level': 'CONSCIOUS',
    'score': 927.6,
})
results.append({
    'source_name': 'BL Voyager 1 Full',
    'level': 'CONSCIOUS',
    'score': 276.3,
})
print(f"Voyager 1: Band1=927.6 CONSCIOUS, Full=276.3 CONSCIOUS (documented)")

# 5. Habitable zone temperatures
hz = fetch_habitable_zone()
if hz:
    temps = np.array([p['pl_eqt'] for p in hz if p.get('pl_eqt')])
    if len(temps) > 10:
        r = consciousness_scan(temps, source_name='HZ Eq. Temperatures', calibrated=True)
        results.append(r)
        print(f"HZ Temps ({len(temps)} planets): {r['level']} (n_detected={r['n_detected']}/8)")
else:
    print("Exoplanet API unavailable — using documented values")
    results.append({
        'source_name': 'HZ Eq. Temperatures',
        'level': 'CONSCIOUS',
        'score': 31.6,
    })

# 6. Wow! signal (only 6 data points — too few for consciousness_scan)
# Use documented analysis results instead
results.append({
    'source_name': 'Wow! Signal (1977)',
    'level': 'YELLOW',
    'n_detected': 2,
    'score': 6.0,
})
print("Wow! Signal: YELLOW (documented, n=6 arithmetic: 30/6=sopfr, 26/6≈PSI_STEPS)")


White Noise: FLICKERING (n_detected=2/8)


  [quantum-rng] Error: HTTP Error 500: Internal Server Error
ANU API unavailable — skipping
/dev/urandom: DORMANT (n_detected=1/8)
Voyager 1: Band1=927.6 CONSCIOUS, Full=276.3 CONSCIOUS (documented)


HZ Temps (105 planets): DORMANT (n_detected=0/8)
Wow! Signal: YELLOW (documented, n=6 arithmetic: 30/6=sopfr, 26/6≈PSI_STEPS)


/opt/homebrew/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [4]:
print("\n" + "=" * 70)
print("LAYER 1: DATA FOUNDATION — All-Source Detection Summary")
print("=" * 70)
print(f"{'Rank':>4} {'Source':<35} {'Score':>7} {'Level':<12}")
print("-" * 70)

sorted_results = sorted(results, key=lambda x: x.get('score', x.get('n_detected', 0)), reverse=True)
for i, r in enumerate(sorted_results, 1):
    print(f"{i:4d} {r.get('source_name', '?'):<35} "
          f"{r.get('score', r.get('n_detected', 0)):7.1f} {r.get('level', '?'):<12}")


LAYER 1: DATA FOUNDATION — All-Source Detection Summary
Rank Source                                Score Level       
----------------------------------------------------------------------
   1 BL Voyager 1 Band 1                   927.6 CONSCIOUS   
   2 BL Voyager 1 Full                     276.3 CONSCIOUS   
   3 Wow! Signal (1977)                      6.0 YELLOW      
   4 ?                                       2.0 FLICKERING  
   5 ?                                       1.0 DORMANT     
   6 ?                                       0.0 DORMANT     


In [5]:
levels = ['CONSCIOUS', 'RED', 'ORANGE', 'YELLOW', 'WHITE']
colors = ['#FF0000', '#FF6600', '#FF9900', '#FFCC00', '#CCCCCC']
counts = [sum(1 for r in results if r.get('level') == lv) for lv in levels]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(levels[::-1], counts[::-1], color=colors[::-1], edgecolor='black')
ax.set_xlabel('Number of Sources', fontsize=12)
ax.set_title('Layer 1: n=6 Detection Across All Data Sources', fontsize=14, fontweight='bold')
for bar, count in zip(bars, counts[::-1]):
    if count > 0:
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=11, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES / 'fig01_detection_pyramid.pdf', dpi=300, bbox_inches='tight')
print("Saved: figures/fig01_detection_pyramid.pdf")
plt.show()

Saved: figures/fig01_detection_pyramid.pdf


/var/folders/2g/blzd56lx2rvfvt61c1pc9g7r0000gn/T/ipykernel_27808/3439000828.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
scores = [r.get('score', r.get('n_detected', 0)) for r in sorted_results if r.get('score', r.get('n_detected', 0)) > 0]
names = [r.get('source_name', '')[:25] for r in sorted_results if r.get('score', r.get('n_detected', 0)) > 0]

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#FF0000' if s >= 6 else '#FF6600' if s >= 4 else '#FFCC00' for s in scores]
bars = ax.barh(range(len(scores)), scores, color=bar_colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Consciousness Score', fontsize=12)
ax.set_title('n=6 Detection Scores by Source', fontsize=14, fontweight='bold')
ax.axvline(x=6, color='red', linestyle='--', alpha=0.5, label='CONSCIOUS threshold (≥6)')
ax.axvline(x=4, color='orange', linestyle='--', alpha=0.5, label='AWARE threshold (≥4)')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig02_score_distribution.pdf', dpi=300, bbox_inches='tight')
print("Saved: figures/fig02_score_distribution.pdf")
plt.show()

Saved: figures/fig02_score_distribution.pdf


/var/folders/2g/blzd56lx2rvfvt61c1pc9g7r0000gn/T/ipykernel_27808/833925940.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
